In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [6]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(
    model=model,
    lr_init=0.001,
    line_search_method="const",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.20141714811325073
epoch:  1, loss: 0.19776757061481476
epoch:  2, loss: 0.1942117065191269
epoch:  3, loss: 0.1906832605600357
epoch:  4, loss: 0.1875779926776886
epoch:  5, loss: 0.18465445935726166
epoch:  6, loss: 0.18143253028392792
epoch:  7, loss: 0.17861774563789368
epoch:  8, loss: 0.17572776973247528
epoch:  9, loss: 0.17285296320915222
epoch:  10, loss: 0.16967575252056122
epoch:  11, loss: 0.16669662296772003
epoch:  12, loss: 0.16478702425956726
epoch:  13, loss: 0.1618739664554596
epoch:  14, loss: 0.15939156711101532
epoch:  15, loss: 0.15718021988868713
epoch:  16, loss: 0.1548580378293991
epoch:  17, loss: 0.1525712013244629
epoch:  18, loss: 0.15013602375984192
epoch:  19, loss: 0.14765383303165436
epoch:  20, loss: 0.14515648782253265
epoch:  21, loss: 0.14276793599128723
epoch:  22, loss: 0.14044281840324402
epoch:  23, loss: 0.1381901502609253
epoch:  24, loss: 0.1359865516424179
epoch:  25, loss: 0.1337919384241104
epoch:  26, loss: 0.13160701096

In [7]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9371787309646606
Test metrics:  R2 = 0.8765665292739868


In [14]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(
    model=model,
    lr=1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.22626128792762756
epoch:  1, loss: 0.2132149040699005
epoch:  2, loss: 0.1203584372997284
epoch:  3, loss: 0.049807362258434296
epoch:  4, loss: 0.03214486315846443
epoch:  5, loss: 0.03036847524344921
epoch:  6, loss: 0.028963832184672356
epoch:  7, loss: 0.02768537402153015
epoch:  8, loss: 0.0265030637383461
epoch:  9, loss: 0.02543669007718563
epoch:  10, loss: 0.021228516474366188
epoch:  11, loss: 0.016310131177306175
epoch:  12, loss: 0.014048690907657146
epoch:  13, loss: 0.011522212065756321
epoch:  14, loss: 0.010229251347482204
epoch:  15, loss: 0.009015992283821106
epoch:  16, loss: 0.008278918452560902
epoch:  17, loss: 0.007628479972481728
epoch:  18, loss: 0.007076545152813196
epoch:  19, loss: 0.006615086458623409
epoch:  20, loss: 0.006281354930251837
epoch:  21, loss: 0.005975417792797089
epoch:  22, loss: 0.0057000755332410336
epoch:  23, loss: 0.005440142005681992
epoch:  24, loss: 0.005224408581852913
epoch:  25, loss: 0.005015517584979534
epoch:

In [15]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9391120076179504
Test metrics:  R2 = 0.6806579232215881


In [16]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(
    model=model,
    lr_init=1,
    line_search_method="interpolate",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.05187857896089554
epoch:  1, loss: 0.05187857896089554
epoch:  2, loss: 0.05187857896089554
epoch:  3, loss: 0.05187857896089554
epoch:  4, loss: 0.05187857896089554
epoch:  5, loss: 0.05187857896089554
epoch:  6, loss: 0.05187857896089554
epoch:  7, loss: 0.05187857896089554
epoch:  8, loss: 0.05187857896089554
epoch:  9, loss: 0.05187857896089554
epoch:  10, loss: 0.05187857896089554
epoch:  11, loss: 0.05187857896089554
epoch:  12, loss: 0.05187857896089554
epoch:  13, loss: 0.05187857896089554
epoch:  14, loss: 0.05187857896089554
epoch:  15, loss: 0.05187857896089554
epoch:  16, loss: 0.05187857896089554
epoch:  17, loss: 0.05187857896089554
epoch:  18, loss: 0.05187857896089554
epoch:  19, loss: 0.05187857896089554
epoch:  20, loss: 0.05187857896089554
epoch:  21, loss: 0.05187857896089554
epoch:  22, loss: 0.05187857896089554
epoch:  23, loss: 0.05187857896089554
epoch:  24, loss: 0.05187857896089554
epoch:  25, loss: 0.05187857896089554
epoch:  26, loss: 0.05

In [17]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -1338.7403564453125
Test metrics:  R2 = -1349.729248046875


In [18]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(
    model=model,
    lr_init=1,
    line_search_method="bisect",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.40136030316352844
epoch:  1, loss: 0.18124011158943176
epoch:  2, loss: 0.12136223167181015
epoch:  3, loss: 0.10230249911546707
epoch:  4, loss: 0.09262148290872574
epoch:  5, loss: 0.05899748578667641
epoch:  6, loss: 0.037440408021211624
epoch:  7, loss: 0.020279401913285255
epoch:  8, loss: 0.016981370747089386
epoch:  9, loss: 0.015082795172929764
epoch:  10, loss: 0.01260261982679367
epoch:  11, loss: 0.01106976717710495
epoch:  12, loss: 0.00962706096470356
epoch:  13, loss: 0.008680030703544617
epoch:  14, loss: 0.007596020586788654
epoch:  15, loss: 0.006926404312252998
epoch:  16, loss: 0.006376671604812145
epoch:  17, loss: 0.006041795015335083
epoch:  18, loss: 0.005828402936458588
epoch:  19, loss: 0.0056992219761013985
epoch:  20, loss: 0.005524187348783016
epoch:  21, loss: 0.005424994044005871
epoch:  22, loss: 0.0053069936111569405
epoch:  23, loss: 0.005260434001684189
epoch:  24, loss: 0.0052007329650223255
epoch:  25, loss: 0.005145490635186434
ep

In [19]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.8652938604354858
Test metrics:  R2 = 0.5300534963607788
